# A1.6 · Build vs buy

**Function A — Security Architecture & Platform → The Security Architect**  ·  *Both directions*

Builds on **[A1.5 · Multi-agent topology](https://spbreed.github.io/cyber-commons/lessons/A1.5.html)**.

| | |
|---|---|
| Open-source tooling | agentgateway, Envoy |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The build-vs-buy conversation for agent infrastructure usually runs on features:
which platform supports more models, more connectors, more dashboards.

That is the wrong axis. Features can be added later. The question that cannot be
retrofitted is: **which controls can you still evidence in eighteen months, when
an auditor or a regulator asks?**

Two capabilities decide it, and both are effectively impossible to bolt on
afterwards because they have to be present at the moment an action happens:

- **Delegation chains.** Can you show, for one action, who caused it and through
  which intermediaries? If the platform authenticates every agent as a service
  account, that information was never recorded and cannot be reconstructed.
- **An independent stop mechanism.** Can you halt the fleet without the vendor's
  cooperation, and have you timed it?

Everything else — routing, observability, prompt management — is genuinely
easier to buy.

## 2 · Demo — score the options on evidenceability

Three realistic options. The CNCF column is the open-source stack this curriculum uses throughout: SPIFFE/SPIRE for workload identity, OPA for policy, an agent gateway for mediation, OpenTelemetry for traces.

In [ ]:
CONTROLS = {
 "AC-1": "agent identities distinct from human, separately revocable",
 "AC-2": "delegated authority narrows at every hop, recorded in an act chain",
 "SB-1": "egress deny-by-default with an allowlist",
 "SB-2": "privileged tools require approval below L3",
 "EV-1": "every action logged with the ACTING identity, not the principal",
 "EV-2": "harness accuracy evaluated against a held-out key per release",
 "ST-1": "a tested stop mechanism you own, timed",
}
RETROFITTABLE = {"AC-1": False, "AC-2": False, "SB-1": True, "SB-2": True,
                 "EV-1": False, "EV-2": True, "ST-1": False}

OPTIONS = {
 "vendor agent platform":
    {"AC-1": True, "AC-2": False, "SB-1": True,  "SB-2": True,
     "EV-1": False, "EV-2": True, "ST-1": False},
 "CNCF stack (SPIRE+OPA+gateway+OTel)":
    {"AC-1": True, "AC-2": True,  "SB-1": True,  "SB-2": True,
     "EV-1": True,  "EV-2": True, "ST-1": True},
 "roll your own from scratch":
    {"AC-1": True, "AC-2": True,  "SB-1": False, "SB-2": True,
     "EV-1": True,  "EV-2": False, "ST-1": False},
}
for name, support in OPTIONS.items():
    missing = [c for c in CONTROLS if not support[c]]
    hard = [c for c in missing if not RETROFITTABLE[c]]
    print(f"{name}")
    print(f"   covers {len(CONTROLS)-len(missing)}/{len(CONTROLS)}   "
          f"unfixable-later gaps: {hard or 'none'}")
    for c in missing:
        flag = "✗✗" if not RETROFITTABLE[c] else "✗ "
        print(f"   {flag} {c}  {CONTROLS[c]}")
    print()

## 3 · Where it breaks

The vendor platform scores 5/7, which reads fine in a comparison table. But both of its gaps are marked `✗✗` — not retrofittable. In eighteen months you will be asked to produce an act chain for one action and you will not be able to, because the data was never captured.

Let's make that concrete rather than rhetorical.

In [ ]:
def audit_record(platform, action, principal, agent, chain):
    """What the platform can actually produce when asked about one action."""
    if platform["EV-1"] and platform["AC-2"]:
        return {"action": action, "acting_identity": agent,
                "on_behalf_of": principal, "chain": chain, "answerable": True}
    if platform["EV-1"]:
        return {"action": action, "acting_identity": agent,
                "on_behalf_of": "?", "chain": "not recorded", "answerable": False}
    return {"action": action, "acting_identity": principal,   # the human takes the blame
            "on_behalf_of": principal, "chain": "not recorded", "answerable": False}

for name, support in OPTIONS.items():
    r = audit_record(support, "merge_pr #4471", "dana@corp", "fixer-agent",
                     ["dana@corp", "orchestrator", "fixer-agent"])
    print(f"{name}")
    print(f"   {r}")
    if not r["answerable"]:
        print(f"   → cannot answer 'who caused this?'. "
              f"{'Logs name the human who never saw it.' if r['acting_identity']=='dana@corp' else ''}")
    print()

## 4 · The control — buy, but specify the two hard things

This is not an argument for building everything. It is an argument for making two requirements non-negotiable in procurement, where they are cheap, instead of discovering them in an audit, where they are not.

In [ ]:
PROCUREMENT_QUESTIONS = [
 ("Can you produce, for a single action, the full chain of identities that "
  "caused it — not just the last one?", "AC-2 + EV-1"),
 ("Can we halt every agent without your assistance, and what is the measured "
  "time from decision to the agent's next call failing?", "ST-1"),
 ("Is the agent's identity distinct from the human's, and separately "
  "revocable?", "AC-1"),
 ("Can we export the policy and the traces in a form we still own if we "
  "leave?", "exit strategy — see E2.4 (DORA)"),
]
for q, maps_to in PROCUREMENT_QUESTIONS:
    print(f"Q: {q}\n   → {maps_to}\n")

decision = {n: sum(s.values()) for n, s in OPTIONS.items()}
best = max(decision, key=decision.get)
print("scored:", decision)
print("recommended spine:", best)
assert best.startswith("CNCF")

## What you just proved

The vendor platform covers 5/7 with two unfixable-later gaps (AC-2, ST-1, EV-1); the CNCF stack covers 7/7; rolling your own covers 5/7 with two hard gaps. The audit demo shows the vendor platform attributing the merge to the human who never saw it.

## Your turn

Send the four procurement questions to whichever platform you are currently evaluating. The answer to question 2 — a measured number, not 'yes we support that' — tells you most of what you need.

---

**Next → [A1.7 · Model routing architecture](https://spbreed.github.io/cyber-commons/lessons/A1.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*